In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

print(cwd)

for candidate in [cwd, *cwd.parents]:
    # print(f"Looking for {marker} in {candidate}")
    if (candidate / marker).exists():
        #print("Project root :", candidate)
        PROJECT_ROOT = str(candidate)
        #print("Project root :", PROJECT_ROOT)
        break


if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)





In [0]:
location=spark.sql(
    """select current_catalog() as current_catalog , current_schema() as current_schema"""
)

type(location)

curr_catalog=location.first()["current_catalog"]
curr_schema=location.first()["current_schema"]


print(curr_catalog)

if curr_catalog !="wsadb1":
    raise RuntimeError("Please run the notebook in the correct catalog ")


# location.show()


In [0]:
display(spark.sql("show catalogs"))

In [0]:
display(spark.sql("show schemas"))

In [0]:
display(spark.sql(f"show schemas in {curr_catalog}"))

In [0]:
cat=curr_catalog
display(spark.sql(f"show schemas in `{cat}`"))

In [0]:
%sql
use catalog system

In [0]:
spark.sql(
    """select current_catalog() as current_catalog , current_schema() as current_schema"""
).show()

In [0]:
%sql
use catalog wsadb1

In [0]:
# create schemas as retail_bronze, retail_silver, retail_gold inside wsadb1
spark.sql(f"create schema if not exists wsadb1.retail_bronze")
spark.sql(f"create schema if not exists wsadb1.retail_silver")
spark.sql(f"create schema if not exists wsadb1.retail_gold")






In [0]:
%sql
use schema retail_bronze

In [0]:
display(
    spark.sql(f"SHOW VOLUMES IN {curr_catalog}.{"retail_bronze"}"))


In [0]:
display(dbutils.fs.ls("/Volumes/wsadb1/retail_bronze/raw_files"))

In [0]:
from pyspark.sql import functions as F

product_csv="dbfs:/Volumes/wsadb1/retail_bronze/raw_files/products.csv"
df = spark.read.format("csv").option("header", "true").load(product_csv)



In [0]:
df.show()

In [0]:
spark.sql("""
           select current_catalog(),
           current_schema()
           """).show()

In [0]:
%sql
use catalog wsadb1;
use schema retail_bronze;

In [0]:
ddl_table='tab_products'
spark.sql(f"drop table if exists {ddl_table}")
spark.sql(f"""
          create table {ddl_table}(
              product_id string,
              product_name string,
              category string,
              subcategory string,
              list_price double,
              cost_price double,
              status string
          )
          
          """)

In [0]:
df.createTempView("products_vw")

In [0]:
%sql
select * from products_vw

In [0]:
spark.sql("""
          select * from wsadb1.retail_bronze.tab_products
          """).show()

In [0]:
spark.sql("""
          insert into wsadb1.retail_bronze.tab_products
          select * from products_vw
          """)

In [0]:
spark.sql("""
          select * from wsadb1.retail_bronze.tab_products
          """).show()

In [0]:
# CTAS
spark.sql("""
          create table if not exists wsadb1.retail_bronze.tab_products_temp
          using delta
          as
          select 
            product_id,
            product_name,
            category,
            subcategory,
            list_price,
            cost_price,
            status
          from
            products_vw
          
          """)

In [0]:
spark.sql("""
          select * from wsadb1.retail_bronze.tab_products_temp
          """).show()

In [0]:
spark.sql("""
          update wsadb1.retail_bronze.tab_products_temp
          set product_name=upper(product_name)
        --    where category='Fashion'
          """)

In [0]:
spark.sql("""
          select * from wsadb1.retail_bronze.tab_products_temp
          """).show()

In [0]:
spark.sql("""
          update wsadb1.retail_bronze.tab_products_temp
          set category=upper(category)
           where category='Fashion'
          """)

In [0]:
spark.sql("""
          select * from wsadb1.retail_bronze.tab_products_temp
          """).show()

In [0]:
spark.sql("""
          delete from wsadb1.retail_bronze.tab_products_temp
           where category='FASHION'
          """)

spark.sql("""
          select * from wsadb1.retail_bronze.tab_products_temp
          """).show()

In [0]:
# Merge

from pyspark.sql import functions as F

customer_csv="dbfs:/Volumes/wsadb1/retail_bronze/raw_files/customers.csv"
customer_df = spark.read.format("csv").option("header", "true").load(customer_csv)

customer_cdc_csv="dbfs:/Volumes/wsadb1/retail_bronze/raw_files/customer_updates_cdc.csv"
customer_cdc_df = spark.read.format("csv").option("header", "true").load(customer_cdc_csv)

In [0]:
customer_df.show() #02-11-2026

In [0]:
customer_cdc_df.show()

In [0]:
customer_df.write.format("delta").mode("overwrite").saveAsTable("wsadb1.retail_gold.customers")

In [0]:
%sql
select * from wsadb1.retail_gold.customers;

In [0]:
customer_cdc_df.createTempView("customer_cdc_df")

In [0]:
merge into target t
using source s
on t.id = s.id
when matched then update set *
when not macthed the insert

In [0]:
spark.sql("""
          merge into wsadb1.retail_gold.customers t
          using customer_cdc_df s
          on t.customer_id = s.customer_id
          when matched and op='U' then 
          update set 
          t.customer_name = s.customer_name,
          t.email = s.email,
          t.city = s.city,
          t.status = s.status
          when not matched and op='I' then 
          insert (customer_id, customer_name, email, city, status) 
          values (s.customer_id, s.customer_name, s.email, s.city, s.status)
          """
          );


# spark.sql("""
#           select * from wsadb1.retail_gold.customers
#           """).show()

In [0]:
# customer_cdc_df.createTempView("customer_cdc_df_vw1")

spark.sql(
"""select  customer_id,customer_name,email,city,status from
          (select c.customer_id, c.customer_name, c.email, c.city, c.status, c.update_ts, row_number() over(partition by c.customer_id order by c.update_ts desc) as rn
          from customer_cdc_df_vw1 c)
           where rn=1
""").show()

In [0]:
spark.sql("""
          merge into wsadb1.retail_gold.customers t
          using 
          (select  customer_id,customer_name,email,city,status,op from
          (select c.customer_id, c.customer_name, c.email, c.city, c.status, c.update_ts, c.op,row_number() over(partition by c.customer_id order by c.update_ts desc) as rn
          from customer_cdc_df_vw1 c)
           where rn=1)
           s
          on t.customer_id = s.customer_id
          when matched and op='U' then 
          update set 
          t.customer_name = s.customer_name,
          t.email = s.email,
          t.city = s.city,
          t.status = s.status
          when not matched and op='I' then 
          insert (customer_id, customer_name, email, city, status) 
          values (s.customer_id, s.customer_name, s.email, s.city, s.status)
          """
          );


# spark.sql("""
#           select * from wsadb1.retail_gold.customers
#           """).show()

In [0]:
spark.sql("""
          select * from wsadb1.retail_gold.customers
           """).show()